# Bonus Challenge Five: Deploying Agents

**Goal:** Demonstrate the ability to deploy and use an agent using Google Agent Platform
(Vertex AI Agent Engine).

**Requirements covered in this notebook:**
1. Create an agent using the ADK -- this reuses the search -> critique -> refine
   workflow (`greeter` / `answer_team`) built in Challenge Four.
2. Deploy the agent to Agent Platform (`agent_engines.create(...)`).
3. Test the deployed agent.

Author: Akhil Sharma (WWT)


In [ ]:
# 1. Install dependencies
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk


In [ ]:
# 2. Imports and configuration
import os
import logging
from typing import Optional

from google.adk.agents import Agent, SequentialAgent
from google.adk.tools import google_search

# --- Configuration ---
# GOOGLE_CLOUD_PROJECT / GOOGLE_CLOUD_LOCATION: your Cloud Skills Boost lab project ID
#   (shown on your Qwiklabs lab page) and a Vertex AI region.
# STAGING_BUCKET: a Cloud Storage bucket (gs://...) Agent Engine uses to stage the
#   deployment package. Create one first if you don't have one, e.g.:
#     gsutil mb -l us-central1 gs://YOUR_PROJECT_ID-agent-engine-staging

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_GCP_PROJECT_ID")
GOOGLE_CLOUD_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
STAGING_BUCKET = os.environ.get("STAGING_BUCKET", "gs://YOUR_PROJECT_ID-agent-engine-staging")

import vertexai
vertexai.init(
    project=GOOGLE_CLOUD_PROJECT,
    location=GOOGLE_CLOUD_LOCATION,
    staging_bucket=STAGING_BUCKET,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("agent_deployment")


## The agent (same workflow as Challenge Four)

`search_agent` -> `critique_agent` -> `refine_agent`, chained in `answer_team`, with
`greeter` as the root entry point. This is the exact agent from Challenge Four -- copied
here so it can be deployed.


In [ ]:
# 3. Search agent: researches the question and drafts an initial answer
SEARCH_AGENT_INSTRUCTIONS = """
You are a research assistant. Use Google Search to find accurate, up-to-date
information that answers the user's question.

Write a clear, well-supported draft answer based on what you find. Keep it
factual and cite specifics (names, numbers, dates) where relevant. This is a
first draft -- a reviewer will critique it next, so it does not need to be
perfect, but it should be accurate and directly address the question.
"""

search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI_FLASH,
    description="Researches the user's question with Google Search and writes a draft answer.",
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    output_key="draft_answer",
    # The built-in google_search tool cannot be combined with any other
    # function-declaration tool (e.g. an auto-injected transfer tool) in the
    # same model call. This agent has no siblings/parent that need it to
    # transfer control, so disable that mechanism defensively.
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)


In [ ]:
# 4. Critique agent: reviews the draft and suggests improvements
CRITIQUE_AGENT_INSTRUCTIONS = """
You are a careful editorial reviewer. You will be shown a draft answer to a
user's question:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

Review it for accuracy, completeness, and clarity. Write a short, specific,
actionable list of suggestions for how to improve it (e.g. missing details,
unclear phrasing, unsupported claims). If the draft is already excellent and
needs no changes, say so explicitly and clearly (e.g. "No changes needed.").

Only output the review notes -- do not rewrite the answer yourself.
"""

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_GEMINI_FLASH,
    description="Reviews the draft answer and suggests concrete improvements.",
    instruction=CRITIQUE_AGENT_INSTRUCTIONS,
    output_key="critique_notes",
)


In [ ]:
# 5. Refine agent: rewrites the draft using the critique
REFINE_AGENT_INSTRUCTIONS = """
You will be shown a draft answer and a reviewer's critique of it:

--- DRAFT ANSWER ---
{draft_answer}
--- END DRAFT ANSWER ---

--- REVIEWER NOTES ---
{critique_notes}
--- END REVIEWER NOTES ---

Rewrite the draft answer, applying the reviewer's suggestions (if the notes
say no changes are needed, just clean up the draft's wording). Output only
the final, polished answer to the user's original question -- no
meta-commentary about the review process.
"""

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_GEMINI_FLASH,
    description="Rewrites the draft answer to incorporate the reviewer's suggested improvements.",
    instruction=REFINE_AGENT_INSTRUCTIONS,
    output_key="final_answer",
)


In [ ]:
# 6. Sequential workflow and root agent
answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Answers a question by researching a draft, critiquing it, and "
        "refining it into a final response."
    ),
    sub_agents=[search_agent, critique_agent, refine_agent],
)

GREETER_INSTRUCTIONS = """
You are the friendly entry point for a question-answering assistant. You do
not answer questions yourself. As soon as the user asks a question, delegate
it to the `answer_team` sub-agent, which will research, critique, and refine
a high-quality response before it is shown to the user.
"""

greeter = Agent(
    name="greeter",
    model=MODEL_GEMINI_FLASH,
    description="Entry point that greets the user and delegates their question to the answer team.",
    instruction=GREETER_INSTRUCTIONS,
    sub_agents=[answer_team],
)


## Step 3: test the agent locally first

Per the workshop's deployment steps, test locally with `AdkApp` *before* deploying --
it's much faster to catch problems here than after a multi-minute deployment.


In [ ]:
# 7. Local helper to run a query and print every event
from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display

def describe_event(event: dict) -> None:
    """
    Print a one-line-per-part summary of a single streamed event: which agent
    authored it, and whether it is text, a tool call, or a tool result.

    Args:
        event (dict): One event dict from an AdkApp/AgentEngine stream_query().
    """
    author = event.get("author", "?")
    content = event.get("content") or {}
    for part in content.get("parts", []) or []:
        if part.get("text"):
            text = part.get("text", "").strip()[:200]
            print(f"  [{author}] TEXT  \u00bb {text}")
        elif part.get("function_call"):
            fc = part["function_call"]
            fc_name = fc.get("name")
            fc_args = fc.get("args")
            print(f"  [{author}] CALL  \u00bb {fc_name}({fc_args})")
        elif part.get("function_response"):
            fr = part["function_response"]
            fr_name = fr.get("name")
            print(f"  [{author}] RESULT \u00bb from {fr_name}")


def ask_agent_verbose(app, question: str, user_id: str = "test-user-id") -> Optional[str]:
    """
    Create a session on the given app/agent, query it once, and print every
    event along the way before returning the final response text. Works for
    both a local AdkApp and a deployed remote AgentEngine, since both expose
    the same create_session/stream_query interface.

    Args:
        app: A local `AdkApp` or a deployed `AgentEngine` (from agent_engines.create()).
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        Optional[str]: The text of the final response, or None on error.
    """
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    event_count = 0
    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=question):
            describe_event(event)
            last_event = event
            event_count += 1
    except Exception as e:
        print(f"Error while querying agent: {e}")
        return None

    if not last_event or "content" not in last_event:
        print(f"Agent did not return a valid final response ({event_count} event(s) received).")
        # Print the raw last event so we can see exactly what came back --
        # e.g. an error_code/error_message, or an empty actions payload --
        # instead of guessing why the stream ended early.
        print("Raw last event:", last_event)
        return None

    return last_event["content"]["parts"][0]["text"]


In [ ]:
# 8. Test locally before deploying
local_app = AdkApp(agent=greeter)

question = "What is the Google Agent Development Kit (ADK) and what is it used for?"
print(f"=== Local test: {question} ===")
local_response = ask_agent_verbose(local_app, question)
print()
display(Markdown(local_response or "*(no response)*"))


## Step 4: deploy the agent to Agent Platform

`agent_engines.create(...)` packages the app and its dependencies, uploads them to the
staging bucket, and provisions a managed, scalable endpoint for it on Vertex AI Agent
Engine. **This is a real cloud deployment and typically takes several minutes** (it
builds a container behind the scenes) -- this cell will block until it finishes.


In [ ]:
# 9. Deploy to Agent Platform (Vertex AI Agent Engine)
from vertexai import agent_engines

remote_agent = agent_engines.create(
    local_app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
    display_name="adk-workshop-answer-team",
    description="Search -> critique -> refine question-answering workflow (Challenge 4/5).",
)

print("Deployed resource name:", remote_agent.resource_name)


## Test the deployed agent

Query the *deployed* agent the same way as the local one -- `ask_agent_verbose` works
unchanged because `AgentEngine` exposes the same `create_session` / `stream_query`
interface as the local `AdkApp`.

Note: the very first request to a freshly deployed agent can occasionally hit a cold
start and return an incomplete stream. If this cell prints "did not return a valid
final response," look at the "Raw last event" it prints, and try simply re-running the
cell once more before assuming something is actually broken.


In [ ]:
# 10. Test the deployed (remote) agent
remote_question = "What is the capital of France, and what is it known for?"
print(f"=== Remote test: {remote_question} ===")
remote_response = ask_agent_verbose(remote_agent, remote_question)
print()
display(Markdown(remote_response or "*(no response)*"))


## Optional cleanup

Agent Engine deployments are billable resources. Uncomment and run the cell below when
you're done grading/testing to delete the deployed agent (leave it commented out until
then so you don't accidentally tear down the deployment before it's reviewed).


In [ ]:
# 11. Optional: delete the deployed agent when you're finished with it
# remote_agent.delete()


## Notes

- Replace the placeholder project/location/staging bucket in the configuration cell
  with your real values. The staging bucket must already exist -- create it with
  `gsutil mb -l <region> gs://<bucket-name>` if needed.
- Deployment can take several minutes; the cell will simply appear to hang while it
  builds and provisions the remote container. That's expected.
- `remote_agent.resource_name` is the full Agent Engine resource ID -- keep it if you
  want to look the deployment up later via `agent_engines.get(resource_name)`.
